# ETF/종목 리스트 앱 - 로직 검증용 (Colab)

`app.py`의 핵심 함수(Streamlit UI 제외)를 그대로 가져와 실제 데이터로 검증합니다.
여기서 확인된 수정사항은 로컬의 `app.py`에 반영합니다.

**순서**: 1) 패키지 설치 → 2) KRX 로그인 → 3) 함수 정의 → 4) 시트별로 하나씩 실행/확인

## 1. 패키지 설치

In [ ]:
!pip install pykrx -q


## 2. KRX 로그인
2026-01-27부터 KRX 정보데이터시스템 조회에 로그인이 필요합니다.

In [ ]:
import os
from getpass import getpass

os.environ["KRX_ID"] = input("KRX 아이디: ")
os.environ["KRX_PW"] = getpass("KRX 비밀번호: ")

KRX 아이디: hanslee9
KRX 비밀번호: ··········


## 2-1 KIS 토큰 발급 관련 셀

In [ ]:
import json
import time as _time
import os

_TOKEN_CACHE_PATH = "/content/.kis_token_cache.json"


def get_kis_access_token(app_key: str, app_secret: str) -> str:
    """KIS Open API 접근토큰 신규 발급 (실제 발급 요청 - 하루 1회만 호출되어야 함)"""
    url = "https://openapi.koreainvestment.com:9443/oauth2/tokenP"
    headers = {"content-type": "application/json"}
    body = {
        "grant_type": "client_credentials",
        "appkey": app_key,
        "appsecret": app_secret,
    }
    resp = requests.post(url, headers=headers, json=body)
    resp.raise_for_status()
    return resp.json()["access_token"]


def get_kis_access_token_cached(app_key: str, app_secret: str) -> str:
    """
    KIS 접근토큰을 캐시해서 재사용한다.
    KIS 정책상 "접근토큰은 1일 1회 발급 원칙"이라 매번 새로 발급받으면 안 됨.
    - 세션 내 재사용: 같은 런타임에서 재호출 시 기존 토큰 그대로 반환
    - 파일 캐시: /content/.kis_token_cache.json에 저장해 Colab 재연결에도 대응
      (단, 완전히 새 런타임이 할당되면 파일도 사라지므로 그때는 새로 발급됨)
    - 23시간 이내 발급된 토큰이면 재사용, 그 이상 지났으면 새로 발급
    """
    now = _time.time()

    # 1) 파일 캐시 확인
    if os.path.exists(_TOKEN_CACHE_PATH):
        try:
            with open(_TOKEN_CACHE_PATH, "r") as f:
                cache = json.load(f)
            if now - cache.get("issued_at", 0) < 23 * 3600:
                print("기존 토큰 재사용 (파일 캐시)")
                return cache["access_token"]
        except Exception:
            pass

    # 2) 캐시 없거나 만료 → 새로 발급 (하루 중 이 경로는 최대 1번만 타야 정상)
    print("새 토큰 발급 요청 중...")
    token = get_kis_access_token(app_key, app_secret)

    with open(_TOKEN_CACHE_PATH, "w") as f:
        json.dump({"access_token": token, "issued_at": now}, f)

    return token


# 토큰 발급 (세션 내 재실행해도 안전 - 캐시 있으면 재사용)
access_token = get_kis_access_token_cached(KIS_APP_KEY, KIS_APP_SECRET)
print("토큰 준비 완료. 앞 10자리:", access_token[:10], "...")

## 3. 공통 함수 정의
app.py와 동일한 로직 (Streamlit 관련 코드만 제거)

In [ ]:
import io
import time
import datetime as dt
import pandas as pd
import requests

ETF_COLS = [
    "종목코드", "ETF명", "시가총액(억원)", "PR\n1개월", "PR\n6개월", "PR\nYTD", "PR\n1년",
    "최근 1년\n분배율",
]

STOCK_COLS_KR = [
    "종목코드", "종목명", "시가총액(억원)", "주가\n1년", "PR\n6개월", "PR\nYTD", "PR\n1년",
    "최근 1년\n배당수익률", "운용사", "브랜드", "자산구분",
]

STOCK_COLS_US = [
    "종목코드", "종목명", "시가총액(Mil)", "주가\n1년", "PR\n6개월", "PR\nYTD", "PR\n1년",
    "최근 1년\n배당수익률", "운용사", "브랜드", "자산구분",
]


def _has_valid_close(df: pd.DataFrame) -> bool:
    if df is None or df.empty or "종가" not in df.columns:
        return False
    return (df["종가"] > 0).any()


def nearest_trading_day_close(get_ohlcv_by_ticker_func, target_date: dt.date, max_back: int = 10, retries: int = 2):
    d = target_date
    for _ in range(max_back):
        date_str = d.strftime("%Y%m%d")
        df = None
        for attempt in range(retries + 1):
            try:
                df = get_ohlcv_by_ticker_func(date_str)
            except Exception:
                df = None
            if _has_valid_close(df):
                return df, date_str
            if attempt < retries:
                time.sleep(0.5)
        d -= dt.timedelta(days=1)
    return pd.DataFrame(), None


def get_etf_ticker_name_safe(ticker: str) -> str:
    """
    pykrx get_etf_ticker_name() 대체 함수.

    원인: 2026-05-27 상장된 단일종목 레버리지/인버스 ETF·ETN 18개 종목이
    KRX 원본 데이터 오류로 ELW 목록에도 중복 등재되어 있음.
    pykrx의 EtxTicker.get_name()은 중복 인덱스일 때 str이 아닌
    pandas.Series를 반환해 str() 변환 시 지저분한 다중행 텍스트가 나옴.

    해결: 중복 발생 시 ETF/ETN 행을 우선 선택하고, 없으면 첫 행을 사용.
    """
    from pykrx.website.krx.etx.ticker import EtxTicker

    df = EtxTicker().df

    if ticker not in df.index:
        return None

    rows = df.loc[[ticker]]  # 리스트 인덱싱으로 항상 DataFrame 형태로 받음

    if len(rows) == 1:
        return str(rows.iloc[0]["종목명"])

    # 중복인 경우: ETF/ETN 카테고리 우선
    preferred = rows[rows["시장"].isin(["ETF", "ETN"])]
    if len(preferred) > 0:
        return str(preferred.iloc[0]["종목명"])

    return str(rows.iloc[0]["종목명"])


def fetch_kis_market_caps(access_token: str, app_key: str, app_secret: str,
                           tickers: list, delay: float = 0.15, verbose: bool = True) -> dict:
    """
    KIS(한국투자증권) Open API로 전 종목 시가총액(억원)을 조회한다.
    '국내주식 현재가 시세' API(FHKST01010100)의 hts_avls 필드가
    이미 억원 단위로 계산된 시가총액이라 별도 계산 불필요.

    KRX 정보데이터시스템 직접 호출과 달리, KIS API는 증권사가
    공식적으로 제공하는 경로라 이용 정책상으로도 적합함(2026-08-27 확인).

    delay: 종목 간 호출 간격(초). 실전 계좌 기준 초당 10~20건 제한이므로
           기본값 0.15초(초당 약 6~7건)로 보수적으로 설정.
    """
    url = "https://openapi.koreainvestment.com:9443/uapi/domestic-stock/v1/quotations/inquire-price"
    headers = {
        "content-type": "application/json",
        "authorization": f"Bearer {access_token}",
        "appkey": app_key,
        "appsecret": app_secret,
        "tr_id": "FHKST01010100",
    }

    market_caps = {}
    total = len(tickers)

    for i, ticker in enumerate(tickers):
        params = {
            "FID_COND_MRKT_DIV_CODE": "J",
            "FID_INPUT_ISCD": ticker,
        }
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=5)
            resp.raise_for_status()
            data = resp.json()
            avls = data.get("output", {}).get("hts_avls")
            market_caps[ticker] = int(avls) if avls not in (None, "") else None
        except Exception:
            market_caps[ticker] = None

        time.sleep(delay)

        if verbose and (i + 1) % 100 == 0:
            print(f"  시가총액 조회 중... {i + 1}/{total}")

    return market_caps


print("OK: 공통 함수 정의 완료 (fetch_kis_market_caps 추가)")

OK: 공통 함수 정의 완료 (fetch_kis_market_caps 추가)


## 4. 1번 시트: 국내 상장 ETF 전체 - 함수 정의

In [ ]:
def build_domestic_etf_sheet(base_date: dt.date, verbose=True) -> pd.DataFrame:
    from pykrx import stock

    def snap(date_obj):
        return nearest_trading_day_close(stock.get_etf_ohlcv_by_ticker, date_obj)

    if verbose: print("기준일 시세 조회 중...")
    today_df, today_str = snap(base_date)
    if today_df.empty:
        raise RuntimeError("기준일 근처에 ETF 시세 데이터가 없습니다. 날짜를 확인해 주세요.")

    if verbose: print("1개월 전 시세 조회 중...")
    m1_df, _ = snap(base_date - dt.timedelta(days=30))

    if verbose: print("6개월 전 시세 조회 중...")
    m6_df, _ = snap(base_date - dt.timedelta(days=182))

    if verbose: print("1년 전 시세 조회 중...")
    y1_df, _ = snap(base_date - dt.timedelta(days=365))

    if verbose: print("연초(YTD) 시세 조회 중...")
    ytd_df, _ = snap(dt.date(base_date.year, 1, 1))

    if verbose: print("종목명 매핑 중...")
    tickers = stock.get_etf_ticker_list(today_str)
    name_map = {}
    for code in tickers:
        try:
            name = get_etf_ticker_name_safe(code)
            name_map[code] = name
        except Exception:
            name_map[code] = None

    if verbose: print(f"시가총액 조회 중 (KIS API, 종목 {len(tickers)}개, 약 3분 소요 예상)...")
    try:
        market_cap_map = fetch_kis_market_caps(
            access_token, KIS_APP_KEY, KIS_APP_SECRET, tickers, verbose=verbose
        )
    except NameError:
        if verbose: print("  KIS 인증 정보(access_token 등)가 없어 시가총액은 결측 처리됩니다.")
        market_cap_map = {}
    except Exception as e:
        if verbose: print(f"  시가총액 조회 실패, 결측 처리됩니다: {e}")
        market_cap_map = {}

    def ret(df_ref, code, close_now):
        if df_ref is None or df_ref.empty or code not in df_ref.index:
            return None
        base_price = df_ref.loc[code, "종가"]
        if not base_price:
            return None
        return round(close_now / base_price - 1, 4)

    rows = []
    for code, row in today_df.iterrows():
        close_now = row["종가"]
        rows.append({
            "종목코드": code,
            "ETF명": name_map.get(code),
            "시가총액(억원)": market_cap_map.get(code),
            "PR\n1개월": ret(m1_df, code, close_now),
            "PR\n6개월": ret(m6_df, code, close_now),
            "PR\nYTD": ret(ytd_df, code, close_now),
            "PR\n1년": ret(y1_df, code, close_now),
            "최근 1년\n분배율": None,
        })

    if verbose: print("완료")
    return pd.DataFrame(rows)[ETF_COLS]

print("OK: build_domestic_etf_sheet 정의 완료 (KIS API 시가총액 연동)")

### 4-1. 실행 및 확인 (오늘 날짜를 최근 영업일로 조정해서 테스트)

In [ ]:
base_date = dt.date(2026, 8, 21)  # 필요시 최근 영업일로 변경

df_etf = build_domestic_etf_sheet(base_date)
print("행 개수:", len(df_etf))
print("컬럼:", list(df_etf.columns))
print("결측치 개수(컬럼별):")
print(df_etf.isna().sum())
df_etf.head(10)

## 5. 2번 시트: KOSPI200 종목 - 함수 정의

In [ ]:
STOCK_COLS_KR = [
    "종목코드", "종목명", "시가총액(억원)", "현재가", "PR\n1개월", "PR\n6개월", "PR\nYTD", "PR\n1년",
    "최근 1년\n배당수익률",
]


def build_kospi200_sheet(base_date: dt.date, verbose=True) -> pd.DataFrame:
    from pykrx import stock

    base_date_str = base_date.strftime("%Y%m%d")

    if verbose: print("KOSPI200 구성종목 조회 중...")
    codes = stock.get_index_portfolio_deposit_file("1028", base_date_str)

    def snap(date_obj):
        return nearest_trading_day_close(
            lambda d: stock.get_market_ohlcv_by_ticker(d, market="ALL"), date_obj
        )

    if verbose: print("기준일 시세 조회 중...")
    today_df, today_str = snap(base_date)
    if today_df.empty:
        raise RuntimeError("기준일 근처에 주식 시세 데이터가 없습니다. 날짜를 확인해 주세요.")

    if verbose: print("1개월 전 시세 조회 중...")
    m1_df, _ = snap(base_date - dt.timedelta(days=30))

    if verbose: print("6개월 전 시세 조회 중...")
    m6_df, _ = snap(base_date - dt.timedelta(days=182))

    if verbose: print("1년 전 시세 조회 중...")
    y1_df, _ = snap(base_date - dt.timedelta(days=365))

    if verbose: print("연초(YTD) 시세 조회 중...")
    ytd_df, _ = snap(dt.date(base_date.year, 1, 1))

    if verbose: print("배당수익률(DIV) 조회 중...")
    try:
        fund_df = stock.get_market_fundamental(today_str, market="ALL")
    except Exception:
        fund_df = pd.DataFrame()

    if verbose: print("시가총액 조회 중...")
    try:
        cap_df = stock.get_market_cap(today_str, market="ALL")
    except Exception:
        cap_df = pd.DataFrame()

    if verbose: print("종목명 매핑 중...")
    name_map = {}
    for code in codes:
        try:
            name_map[code] = stock.get_market_ticker_name(code)
        except Exception:
            name_map[code] = None

    def ret(df_ref, code, close_now):
        if df_ref is None or df_ref.empty or code not in df_ref.index:
            return None
        base_price = df_ref.loc[code, "종가"]
        if not base_price:
            return None
        return round(close_now / base_price - 1, 4)

    rows = []
    for code in codes:
        if code not in today_df.index:
            continue
        close_now = today_df.loc[code, "종가"]

        pr1m = ret(m1_df, code, close_now)
        pr6m = ret(m6_df, code, close_now)
        pr_ytd = ret(ytd_df, code, close_now)
        pr1y = ret(y1_df, code, close_now)

        div_yield = None
        if not fund_df.empty and code in fund_df.index:
            div_val = fund_df.loc[code, "DIV"]
            div_yield = round(div_val / 100, 4) if pd.notna(div_val) else None

        market_cap = None
        if not cap_df.empty and code in cap_df.index:
            cap_val = cap_df.loc[code, "시가총액"]
            market_cap = round(cap_val / 1e8) if pd.notna(cap_val) else None

        rows.append({
            "종목코드": code,
            "종목명": name_map.get(code),
            "시가총액(억원)": market_cap,
            "현재가": close_now,
            "PR\n1개월": pr1m,
            "PR\n6개월": pr6m,
            "PR\nYTD": pr_ytd,
            "PR\n1년": pr1y,
            "최근 1년\n배당수익률": div_yield,
        })

    if verbose: print("완료")
    out = pd.DataFrame(rows)[STOCK_COLS_KR]
    out = out.sort_values("시가총액(억원)", ascending=False, na_position="last").reset_index(drop=True)
    return out

### 5-1. 실행 및 확인

In [ ]:
df_kospi200 = build_kospi200_sheet(base_date)
print("행 개수:", len(df_kospi200), "(200종목이어야 정상)")
print("결측치 개수(컬럼별):")
print(df_kospi200.isna().sum())
df_kospi200.head(10)

In [ ]:
df_sp500_test = process_us_stock_sheet("S&P500 테스트", get_sp500_tickers, limit=10)
df_sp500_test

## 6. 4, 5, 6번 시트: 미국 종목 - 함수 정의

In [ ]:
STOCK_COLS_US = [
    "종목코드", "종목명", "시가총액(Mil)", "현재가", "PR\n1개월", "PR\n6개월", "PR\nYTD", "PR\n1년",
    "최근 1년\n배당수익률",
]

_WEB_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def _read_html_tables(url: str):
    import requests
    resp = requests.get(url, headers=_WEB_HEADERS, timeout=15)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))


def get_sp500_tickers() -> pd.DataFrame:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = _read_html_tables(url)
    df = tables[0][["Symbol", "Security"]].rename(
        columns={"Symbol": "종목코드", "Security": "종목명"}
    )
    df["종목코드"] = df["종목코드"].str.replace(".", "-", regex=False)
    return df


def get_nasdaq100_tickers() -> pd.DataFrame:
    url = "https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies"
    tables = _read_html_tables(url)
    for t in tables:
        cols = [str(c) for c in t.columns]
        if any("Ticker" in c for c in cols) or any("Symbol" in c for c in cols):
            ticker_col = [c for c in cols if "Ticker" in c or "Symbol" in c][0]
            name_col = [c for c in cols if "Company" in c or "Security" in c]
            name_col = name_col[0] if name_col else cols[0]
            df = t[[name_col, ticker_col]].rename(
                columns={name_col: "종목명", ticker_col: "종목코드"}
            )
            return df
    raise ValueError("나스닥100 표를 찾지 못했습니다.")


def _get_schd_nport_xml_url() -> str:
    """SCHWAB STRATEGIC TRUST(CIK 1454889)의 최신 NPORT-P 필링들 중
    SCHD 시리즈(S000034163)에 해당하는 필링의 primary_doc.xml URL을 찾는다."""
    import requests

    cik = "0001454889"
    cik_int = str(int(cik))
    sec_headers = {"User-Agent": "ETF-List-App research-tool@example.com"}

    resp = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=sec_headers, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    forms = data["filings"]["recent"]["form"]
    accessions = data["filings"]["recent"]["accessionNumber"]
    nport_filings = [a for f, a in zip(forms, accessions) if f == "NPORT-P"]

    for accession in nport_filings[:20]:
        accession_nodash = accession.replace("-", "")
        xml_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_nodash}/primary_doc.xml"
        r = requests.get(xml_url, headers=sec_headers, timeout=15)
        if r.status_code == 200 and "S000034163" in r.text:
            return xml_url

    raise RuntimeError("SEC EDGAR에서 SCHD의 최신 N-PORT 필링을 찾지 못했습니다.")


def _cusip_to_ticker_batch(cusip_list, batch_size=10) -> dict:
    """OpenFIGI API(무료, 키 불필요)로 CUSIP 리스트를 티커로 일괄 변환."""
    import requests
    import time

    results = {}
    for i in range(0, len(cusip_list), batch_size):
        batch = cusip_list[i:i + batch_size]
        jobs = [{"idType": "ID_CUSIP", "idValue": c} for c in batch]
        resp = requests.post(
            "https://api.openfigi.com/v3/mapping",
            json=jobs,
            headers={"Content-Type": "application/json"},
            timeout=20,
        )
        if resp.status_code == 200:
            data = resp.json()
            for cusip, item in zip(batch, data):
                if "data" in item and item["data"]:
                    us_match = next((d for d in item["data"] if d.get("exchCode") == "US"), None)
                    chosen = us_match or item["data"][0]
                    results[cusip] = chosen.get("ticker")
                else:
                    results[cusip] = None
        else:
            for c in batch:
                results[c] = None
        time.sleep(0.3)
    return results


def get_schd_holdings() -> pd.DataFrame:
    """SCHD(Schwab US Dividend Equity ETF) 구성종목.
    SEC EDGAR의 Form N-PORT(공식 규제 공시)에서 전체 보유종목(CUSIP 기준)을 가져오고,
    OpenFIGI API로 CUSIP을 실제 매매 티커로 변환한다."""
    xml_url = _get_schd_nport_xml_url()

    import requests
    import xml.etree.ElementTree as ET

    sec_headers = {"User-Agent": "ETF-List-App research-tool@example.com"}
    r = requests.get(xml_url, headers=sec_headers, timeout=15)
    r.raise_for_status()
    root = ET.fromstring(r.text)

    holdings = []
    for elem in root.iter():
        tag = elem.tag.split("}")[-1]
        if tag == "invstOrSec":
            item = {}
            for child in elem:
                ctag = child.tag.split("}")[-1]
                if ctag in ("name", "cusip"):
                    item[ctag] = child.text
            holdings.append(item)

    cusips = [h.get("cusip") for h in holdings if h.get("cusip") and h.get("cusip") != "000000000"]
    ticker_map = _cusip_to_ticker_batch(cusips)

    rows = []
    for h in holdings:
        cusip = h.get("cusip")
        if not cusip or cusip == "000000000":
            continue
        ticker = ticker_map.get(cusip)
        if not ticker:
            continue
        rows.append({"종목코드": ticker, "종목명": h.get("name")})

    if not rows:
        raise RuntimeError("SCHD 구성종목을 SEC N-PORT에서 파싱했으나 결과가 비어 있습니다.")

    return pd.DataFrame(rows)


def calc_us_stock_metrics(ticker: str) -> dict:
    """yfinance로 미국 종목의 시가총액/주가/PR/배당수익률 계산."""
    import yfinance as yf

    try:
        t = yf.Ticker(ticker)
        hist = t.history(period="2y", auto_adjust=False)
        if hist.empty:
            return {}

        hist = hist.sort_index()
        last_close = hist["Close"].iloc[-1]
        last_date = hist.index[-1]

        def price_n_days_ago(days):
            target = last_date - pd.Timedelta(days=days)
            sub = hist[hist.index <= target]
            return sub["Close"].iloc[-1] if not sub.empty else None

        def price_at_ytd_start():
            ytd_start = pd.Timestamp(year=last_date.year, month=1, day=1, tz=hist.index.tz)
            sub = hist[hist.index <= ytd_start]
            if not sub.empty:
                return sub["Close"].iloc[-1]
            sub2 = hist[hist.index >= ytd_start]
            return sub2["Close"].iloc[0] if not sub2.empty else None

        p1m = price_n_days_ago(30)
        p6m = price_n_days_ago(182)
        p1y = price_n_days_ago(365)
        p_ytd = price_at_ytd_start()

        def pr(base):
            if base is None or base == 0:
                return None
            return round(last_close / base - 1, 4)

        div_yield = None
        try:
            divs = t.dividends
            if divs is not None and not divs.empty:
                one_year_ago = last_date - pd.Timedelta(days=365)
                recent_divs = divs[divs.index >= one_year_ago]
                div_yield = round(float(recent_divs.sum()) / last_close, 4)
        except Exception:
            pass

        # 시가총액: fast_info 우선 사용 (info보다 안정적), 실패 시 info로 폴백
        market_cap = None
        try:
            fi = t.fast_info
            raw_cap = fi.get("marketCap") or fi.get("market_cap")
            if raw_cap:
                market_cap = round(raw_cap / 1e6)
        except Exception:
            pass
        if market_cap is None:
            try:
                info = t.info
                raw_cap = info.get("marketCap")
                market_cap = round(raw_cap / 1e6) if raw_cap else None
            except Exception:
                pass

        return {
            "시가총액(Mil)": market_cap,
            "현재가": round(float(last_close), 2),
            "PR\n1개월": pr(p1m),
            "PR\n6개월": pr(p6m),
            "PR\nYTD": pr(p_ytd),
            "PR\n1년": pr(p1y),
            "최근 1년\n배당수익률": div_yield,
        }
    except Exception:
        return {}


def process_us_stock_sheet(label, ticker_df_func, limit=None) -> pd.DataFrame:
    df = ticker_df_func()
    if limit:
        df = df.head(limit).reset_index(drop=True)
    rows = []
    n = len(df)
    for i, r in df.iterrows():
        metrics = calc_us_stock_metrics(r["종목코드"])
        rows.append({**r.to_dict(), **metrics})
        if (i + 1) % 20 == 0 or (i + 1) == n:
            print(f"{label} {i + 1}/{n}")
        time.sleep(0.05)
    out = pd.DataFrame(rows)
    for c in STOCK_COLS_US:
        if c not in out.columns:
            out[c] = None
    out = out[STOCK_COLS_US]
    out = out.sort_values("시가총액(Mil)", ascending=False, na_position="last").reset_index(drop=True)
    return out

print("OK: 미국 종목 함수 정의 완료 (fast_info로 시가총액 안정성 개선, 운용사/브랜드/자산구분 삭제, 현재가/PR1개월 반영)")

In [ ]:
df_sp500_test = process_us_stock_sheet("S&P500 테스트", get_sp500_tickers, limit=10)
df_sp500_test

### 6-1. 소규모 테스트 (S&P500 중 앞 10개만) — 로직 검증용, 빠르게 확인
전체 500/100종목 검증은 로직에 문제 없는 게 확인된 후 `limit=None`으로 돌리세요.

In [ ]:
df_sp500 = process_us_stock_sheet("S&P500", get_sp500_tickers)
print("행 개수:", len(df_sp500), "(500종목 근처여야 정상)")
print("결측치 개수(컬럼별):")
print(df_sp500.isna().sum())
df_sp500.head(10)

In [ ]:
import requests
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

resp = requests.get(url, headers=headers)
print("status code:", resp.status_code)

tables = pd.read_html(resp.text)
print("테이블 개수:", len(tables))
print(tables[0].columns.tolist())
tables[0].head()

In [ ]:
df_nasdaq100_sample = process_us_stock_sheet("나스닥100(샘플)", get_nasdaq100_tickers, limit=10)
df_nasdaq100_sample

In [ ]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/Nasdaq-100"
resp = requests.get(url, headers=_WEB_HEADERS, timeout=15)
print("status:", resp.status_code)

tables = pd.read_html(io.StringIO(resp.text))
print("테이블 개수:", len(tables))

for i, t in enumerate(tables):
    print(f"\n--- 테이블 {i} ---")
    print("컬럼:", list(t.columns))
    if len(t) > 0:
        print(t.head(2))

In [ ]:
df_schd_sample = process_us_stock_sheet("SCHD(샘플)", get_schd_holdings, limit=10)
df_schd_sample

In [ ]:
import requests

url = "https://www.schwabassetmanagement.com/allholdings/SCHD"
resp = requests.get(url, headers=_WEB_HEADERS, timeout=15)
print("status:", resp.status_code)
print("content length:", len(resp.text))
print(resp.text[:1000])

In [ ]:
url2 = "https://stockanalysis.com/etf/schd/holdings/"
resp2 = requests.get(url2, headers=_WEB_HEADERS, timeout=15)
print("status:", resp2.status_code)
print("length:", len(resp2.text))

tables2 = pd.read_html(io.StringIO(resp2.text))
print("테이블 개수:", len(tables2))
for i, t in enumerate(tables2):
    print(f"\n--- 테이블 {i} ---")
    print("컬럼:", list(t.columns))
    print(t.head(3))

### 6-2. (선택) 전체 종목 검증 — 시간이 걸립니다
샘플 테스트에서 문제 없었을 때만 실행하세요.

In [ ]:
df_sp500_full = process_us_stock_sheet("S&P500", get_sp500_tickers, limit=None)
df_nasdaq100_full = process_us_stock_sheet("나스닥100", get_nasdaq100_tickers, limit=None)
df_schd_full = process_us_stock_sheet("SCHD", get_schd_holdings, limit=None)

## 7. 결과 엑셀로 다운로드 (선택)
검증한 데이터를 엑셀로 받아서 직접 눈으로 확인하고 싶을 때 사용하세요.

In [ ]:
# 검증한 데이터를 엑셀로 받아서 직접 눈으로 확인하고 싶을 때 사용하세요.

from google.colab import files

sheets_to_export = {
    "국내 상장 ETF": df_etf,
    "KOSPI200종목": df_kospi200,
    "미국S&P500종목": df_sp500_full,
    "미국나스닥100종목": df_nasdaq100_full,
    "미국 배당 ETF(SCHD)": df_schd_full,
}

out_path = "검증결과.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    for name, df in sheets_to_export.items():
        if df is not None:
            df.to_excel(writer, sheet_name=name[:31], index=False)

files.download(out_path)